# MENTOR

Proposes negative control outcomes (NCOs) for a target–comparator drug pair using your OMOP CDM.

**Before running:** fill in `.env` (database) and set your vLLM endpoint in `src/core/config/experiment.py` (`MODELS`). See `README.md`.

## 1. Settings

Drugs are given as OMOP standard ingredient concept IDs (RxNorm). The example below is one of the drug pairs evaluated in the paper.

In [ ]:
ingredient_concept_ids = [
    45774751,   # empagliflozin
    1580747,    # sitagliptin
]
name = "empagliflozin_sitagliptin"   # results go to logs/<time>_<name>/

# Same setting as the paper: a single pass at 10,000 patients.
# To let the Diagnostician lower the threshold, use e.g. min_patients=70000, min_floor=10000.
overrides = {
    "search": {"min_patients": 10000, "min_floor": 10000, "step": 10000},
}

## 2. Run

Progress is printed below and saved to `logs/<time>_<name>/run.log`. In the paper (two RTX A6000 GPUs), a run took about 53 minutes per pair.

In [ ]:
from main import run_experiment

result = run_experiment(ingredient_concept_ids, name, overrides=overrides)
print(f"status={result['status']}  nco_count={result['nco_count']}  folder={result['run_dir']}")

## 3. Results

`decided_by` is `cem+llm agree` when CEM and the first-pass LLM agreed, or `review` when the Adjudicator decided. The same folder also holds `concept_set.json` (ATLAS) and `negative_control_outcomes.csv` (CohortGenerator/Strategus).

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 200)
ncos = pd.read_csv(result["run_dir"] / "nco_list.csv")
print(ncos["decided_by"].value_counts().to_string())
ncos